# Ingénierie des caractéristiques I

Ce notebook implémente les étapes demandées pour la **Partie 2** du Jalon 2

## 1. Organisation et partitionnement des données

Nous utilisons le fichier `data/processed/shots.csv` généré à l'Étape 1 qui contient déjà les tirs et buts extraits pour toutes les saisons 2016-2017 à 2023-2024.

Comme il est demandé, nous allons d'abord séparé les données en 3 ensembles fixes. 
- Test : On réserve la saison 2023/2024 pour les tests finaux
- Entraînement : On prend 80% dans les saisons restantes 
- Validation : On prend 20% dans les saisons restantes

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from ift6758.data_acquisition import PROJECT_ROOT
from ift6758.features.build_dataset import PROCESSED


data_path = PROJECT_ROOT / "data" / "processed" / "shots.csv"
df_all = pd.read_csv(data_path)

df_test_regular = df_all[(df_all["game_type"] == 2) & (df_all["season"] == 20232024)]
df_test_playoffs = df_all[(df_all["game_type"] == 3) & (df_all["season"] == 20232024)]

df_regular_dev = df_all[(df_all["game_type"] == 2) & (df_all["season"] != 20232024)]

# Ici on sépare les saisons réservées au développement en deux ensemble, on 
# doit s'assurer que les ensembles soient stratifiés sur is_goal pour ne pas 
# avoir de déséquilibre.
train_df, val_df = train_test_split(
    df_regular_dev,
    test_size=0.20,
    stratify=df_regular_dev['is_goal'],
    random_state=42
)

print("Proportion buts (Train) :", train_df['is_goal'].mean())
print("Proportion buts (Val)   :", val_df['is_goal'].mean())

PROCESSED.mkdir(parents=True, exist_ok=True)

csv_file = PROCESSED / 'test_regular_20232024.csv'
df_test_regular.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'test_playoffs_20232024.csv'
df_test_playoffs.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'train_set.csv'
train_df.to_csv(csv_file, index=False)

csv_file = PROCESSED / 'validation_set.csv'
val_df.to_csv(csv_file, index=False)

Dans notre jeu de données train_df nous avons toutes les caractéristiques que build_dataset inclus. Autrement dit nous respectons avoir les caractéristiques shot_distance, shot_angle, is_goal, empty_net.
Pour le reste de la section, nous allons définir un dataframe qui contient simplement ces colonnes. On va aussi afficher les valeurs manquantes de chaque colonnes :

In [ ]:
df_cols_imp_train = train_df[["shot_distance", "shot_angle", "is_goal", "empty_net"]].copy()

# On va transformer empty_net pour que ce soit 0 ou 1 au lieu de True or False
df_cols_imp_train["empty_net"] = df_cols_imp_train["empty_net"].map({True: 1, False: 0})

print("Nombre de valeurs manquantes par caractéristique :")
print(df_cols_imp_train.isna().sum())


print(df_cols_imp_train.head(10))

On remarque que nous avons 97 données avec la distance manquante, 2354 avec l'angle manquant puis 3242 pour empty_net.

## 2. Premières figures du nombre de tirs en fonction de différentes variables

Maintenant, nous allons faire l'histogramme du nombre de tirs (buts et non-buts séparés), regroupés (binned) par distance. Après avoir essayé l'échelle normale, on voit que l'ordre de grandeur diffère beaucoup selon le résultats du tirs, dans ce cas le nombre de tirs subira une transformation logarithmique.

In [ ]:
# Pour que ce soit plus clair on va mettre la légende de façon explicite
df_cols_imp_train["goal_status"] = df_cols_imp_train["is_goal"].map({0:"Non but", 1:"But"})
# On déclare une palette de couleur
colors = {"Non but": "#4C72B0", "But": "#C44E52"}
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=df_cols_imp_train,
    x="shot_distance",
    hue="goal_status",
    palette=colors,
    binwidth=5.0,
    binrange=(0, 200),
    element="step",
    ax=ax
)
ax.set_yscale("log")
ax.set_title("Histogramme du nombre de tirs regroupés sur la distance selon le type (is_goal)")
ax.set_xlabel("Distance (pieds)")
ax.set_ylabel("Nombre de tirs (échelle log)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

On voit qu'il y a énormément de tirs plus loin que 100 pieds, ce qui n'est pas anormal, mais il faudra être prudent. Des tirs qui sont des buts passé 100 pieds lorsque ce n'est pas en filet désert c'est plutôt rare. Peut-être que la fonction qui calcule le defending side s'est trompé, ce sera à investiguer. 

Maintenant, nous allons faire le même graphique mais regroupé par l'angle, nous allons aussi mettre l'échelle logarithmique.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=df_cols_imp_train,
    x="shot_angle",
    hue="goal_status",
    palette=colors,
    binwidth=5.0,
    binrange=(0, 90),
    element="step",
    ax=ax
)
ax.set_yscale("log")
ax.set_title("Histogramme du nombre de tirs regroupés sur l'angle selon le type (is_goal)")
ax.set_xlabel("Angle (degrés)")
ax.set_ylabel("Nombre de tirs (échelle log)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

On remarque que nous avons l'impression que plus le tir est décentré au filet plus le pourcentage de but semble descendre.

Nous allons maintenant faire l'histogramme en 2D ou un axe est la distance et l'autre l'angle. Nous utiliserons l'échelle logarithmique pour les couleurs aussi puisqu'il y a beaucoup plus de tirs proche du filet avec un angle de 0 que de tirs loins avec des angles élevés.

In [ ]:
g = sns.jointplot(
    data=df_cols_imp_train,
    x="shot_distance",
    y="shot_angle",
    kind="hist",
    xlim=(0, 200),
    ylim=(0,90),
    height=8,
    # Fait des bins 5pieds*5degrés et ajoute la barre de couleur, LogNorm 
    # permet que l'échelle de couleur soit logarithmique. Vmin et Vmax doit 
    # être donné sinon le graphique ne s'affiche pas
    joint_kws={"binwidth": (5, 5), "cbar": True, "norm": LogNorm(), "vmin": None, "vmax": None, "cbar_kws": {"label": "Nombre de tirs (échelle log)"}},
    # Fait les même pas pour les hist marginaux
    marginal_kws={"binwidth": 5},
)
# Applique transformation log aux histogrammes marginaux
g.ax_marg_x.set_yscale("log")
g.ax_marg_y.set_xscale("log")
g.set_axis_labels("Distance (pieds)", "Angle (degrés)")
g.figure.suptitle("Histogramme 2D du nombre de tirs selon la distance et l'angle", y=1.02)
plt.show()

On remarque les mêmes résultats que nous avions vu avec les deux derniers histogrammes. Les tirs sont principalement concentrés dans des petites distances et angles.

## 3. Figures en fonction  #buts / (#non_buts + #buts)

Nous allons commencer par faire la figure du taux de tirs en fonction de la distance : 

In [ ]:
# On rajoute la catégorie de bins associée à la distance
df_cols_imp_train["bins_distance"] = pd.cut(df_cols_imp_train["shot_distance"], 
                                            bins= list(range(0, 205, 5)), 
                                            include_lowest=True
                                            ).apply(lambda x: x.mid if pd.notna(x) else np.nan)

rate_per_dist = (df_cols_imp_train.groupby("bins_distance", observed=True)["is_goal"].agg(["mean", "count"]).reset_index())
print(rate_per_dist)

fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=rate_per_dist,
    x="bins_distance",
    y="mean",
    marker="o",
    color="#C44E52",
    linewidth=2,
    ax=ax
)
ax.set_title("Graphique du taux de buts selon la distance")
ax.set_xlabel("Distance du tir (pieds)")
ax.set_ylabel("Taux de buts")
ax.set_ylim(bottom=0)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

On remarque que le taux chute en fonction de la distance ce qui est très logique, mais celui-ci commence à remonter lorsque la disrtance remonte au dessus de 80-90 pieds vers 4-5%. C'est potentiellement puisque ce sont des tirs en filets désert ou causé par des données aberrantes.

Nous allons maintenant faire la figure du taux de tirs en fonction de l'angle : 

Après avoir testé un binning de [3,5,10,15] nous avons choisi 5, c'est l'angle parfait pour avoir un bon niveau de détails.

In [ ]:
# On rajoute la catégorie de bins associée à la distance
df_cols_imp_train["bins_angles"] = pd.cut(df_cols_imp_train["shot_angle"], 
                                            bins= list(range(0, 95, 5)), 
                                            include_lowest=True
                                            ).apply(lambda x: x.mid if pd.notna(x) else np.nan)

rate_per_angle = (df_cols_imp_train.groupby("bins_angles", observed=True)["is_goal"].agg(["mean", "count"]).reset_index())
print(rate_per_angle)

fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=rate_per_angle,
    x="bins_angles",
    y="mean",
    marker="o",
    color="#C44E52",
    linewidth=2,
    ax=ax
)
ax.set_title("Graphique du taux de buts selon l'angle")
ax.set_xlabel("Angle du tir (valeur absolue)")
ax.set_ylabel("Taux de buts")
ax.set_ylim(bottom=0)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

Nous avons choisi de prendre l'angle en valeur absolue initialement puisque, dans notre situation, le côté gauche ou droit importe peu. Du point de vue hockey c'est important puique un joueur gaucher est mieux positionné à droite pour faire un one-timer, toutefois comme nous n'avons pas le type de tir et le côté dominant du joueur cela ne change rien.

La discussion et les interprétations seront dans le blog. 

## 3. Vérification du sens des données


Nous allons faire l'histogramme du nombre de buts regroupés par distance, et séparé filet-désert / non filet-désert

In [ ]:
# Nous allons isoler uniquement les buts
df_goals = df_cols_imp_train[(df_cols_imp_train["is_goal"] == 1)].reset_index()
# Pour que ce soit plus clair on va mettre la légende de façon explicite
df_goals["goaltender_status"] = df_goals["empty_net"].map({0:"Filet avec gardien", 1:"Filet désert"})
# On déclare une palette de couleur
colors = {"Filet avec gardien": "#4C72B0", "Filet désert": "#C44E52"}
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=df_goals,
    x="shot_distance",
    hue="goaltender_status",
    palette=colors,
    binwidth=5.0,
    binrange=(0, 200),
    element="step",
    ax=ax
)
ax.set_yscale("log")
ax.set_title("Histogramme du nombre de buts regroupés sur la distance selon le type (goaltender_status)")
ax.set_xlabel("Distance (pieds)")
ax.set_ylabel("Nombre de buts (échelle log)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Hypothèses** : il est normal que dans notre zone défensive, il y ait des buts en filet désert. Toutefois, on remarque des bosses entre 110 pieds et 140 pieds de buts qui ne sont pas fait en filet désert anormalement haut, même chose pour une bosse entre 150 et 180 pieds et entre 185 et 190. Connaissant le hockey, il est impossible qu'il y ait eu autant de buts en zone défensive avec un gardien. Il y a nécessairement une caractéristique qui cloche, nous allons devoir investiguer cela.

Nous allons d'abord filtrer sur le tain_df, nous allons isoler les événements qui ont une distance plus grande que 100 pieds, avec un but qui n'est pas en filet désert. Ensuite, nous allons regarder les caractéristiques de ces événements qui pourraient trahir une erreur flagrante. 

In [ ]:
anomalies = train_df[(train_df["shot_distance"] > 100) & (train_df["is_goal"] == 1) & (train_df["empty_net"] == False)]
print(f"Nombre de cas suspects : {len(anomalies)}")

# On va garder les colonnes les plus pertinentes pour détecter une anomalie : 
anomalies[['game_id', 'game_date', 'team_name', 'period', 'time_in_period', 'shooter_name', 'goalie_name', 'shot_type', 'x_coord', 'y_coord', 'shot_distance']].head(10)

Les évidences les plus révélatrices sont les types de tirs "tip-in". Un "tip-in" est un tir qui consiste à toucher légèrement la rondelle avec son bâton pour changer sa trajectoire alors qu'elle se dirige vers le filet. Ça ne fait donc aucun sens qu'un joueur ait marqué un but à plus de 100 pieds en faisant un tip-in. Prenons par exemple, le but marqué par Brayden Point en tip-in à 164 pieds contre le gardien James Reimer (game_id : 2017020174), nous allons confirmer notre hypothèse en regardant le tir en rediffusion sur le gamecenter de la NHL. Le but n'était pas disponible sur le site, mais sur la chaîne youtube officielle de la NHL il est possible de voir le but en question à 1min20 sec de la vidéo : https://www.youtube.com/watch?v=IyF1_4X618A. On voit clairement que le but se fait à quelques pieds du but et non à 164 pieds. Ce qui prouve qu'il y a des erreurs de position dans l'API de la NHL, probablement que la position a été prise à l'envers. Si on transforme la position en X de -75 en 75 alors la distance devient réaliste. Bref, puisque nous ne faisons pas une analyse des anomalies plus poussée nous allons garder ces données, mais il faut garder en tête que plusieurs buts fait avec un gardien passé 100 pieds sont probablement erronés. 